# 1. 模式四：@dataclass

@dataclass是 Python 标准库 dataclasses 提供的类装饰器，用于简化“以字段为核心”的数据类定义。

给类加上 @dataclass 后，Python 会根据字段声明自动生成常用方法，例如：

* __init__
* __repr__
* __eq__

因此，从对象行为上看， @dataclass 创建的类，常常 近似于 手写这些方法的普通类。

In [1]:
from dataclasses import dataclass
@dataclass
class Movie:
    title: str
    year: int
    director: str
    rating: float

这类写法的主要价值是：让“数据结构定义”更简洁、更清晰。

注意：

1. 虽然 @dataclass 在行为上接近于手写 __init__ 、 __repr__ 、 __eq__ 的普通类，但二者**并不完全等价**。
2. @dataclass 修饰后的类仍然是普通的python类，但它会被标准库标记为数据类，并携带dataclass的字段元信息，手写 __init__ 等方法的普通类不能替代@dataclass修饰的类。

前者可以作为Langchain的Schema，而后者不行。


举例：


模型的初始化：

In [4]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

print(model.invoke("你好!"))

content='你好！有什么我可以帮你处理的？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 4390, 'total_tokens': 4404, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 3840, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'resp_0f0c47d37bfefe46016ab91083cc1487d29bfb87d7657b8814', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0e2e8-6810-7cf3-b803-28f6068b0f08-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 4390, 'output_tokens': 14, 'total_tokens': 4404, 'input_token_details': {'cache_read': 3840}, 'output_token_details': {}}


In [3]:
from dataclasses import dataclass
from pydantic import Field

@dataclass
class Movie():
    """
    电影的详细信息
    """

    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

structured_model = model.with_structured_output(Movie)
response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'director': '克里斯托弗·诺兰', 'rating': 8.8, 'title': '盗梦空间', 'year': 2010}
<class 'dict'>


# 2. 使用输出解析器(不推荐)

上面是4种输出结构化输出的方式，我们也可以采用输出解析器的方式

这种方法更传统，依赖于在提示词中明确指示模型输出特定格式的文本，然后使用解析器进行转换。

其流程是： 提示词指导 (引导生成指定类型) → 模型生成文本 → 解析器转换 。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# 1. 创建提示词模板

prompt_template = ChatPromptTemplate.from_messages([
    ("system","回答用户问题,必须始终输出一个包含title(电影标题)和year(上映年份)的JSON 对象"),
    ("human","问题：{question}")
])

import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)
# 5. 创建链
# 这里的`|` 是 LangChain 的 LCEL（LangChain Expression Language）管道运算符 ，它能让多个组件「首尾相连」，数据从左到右依次流过。
# 第一个输出进入第二个输入
# LangChain 的`Runnable` 基类就重写了`__or__` 
# class Runnable:
#     def __or__(self, other):
#         # 把 self 和 other 包装成一个新的"顺序执行"对象
#         return RunnableSequence(self, other)
# a | b # 实际调用 a.__or__(b) → RunnableSequence(a, b) 
# a | b | c # 从左往右结合 → RunnableSequence(a, b, c)
chain = prompt_template | model | parser
# 6. 调用（返回字典）
response = chain.invoke({"question": "介绍电影《盗梦空间》"})
#
# response =parser.invoke(model.invoke(prompt_template.invoke({"question":"介绍电影《盗梦空间》"}))) 模型输出后再绑定解析器，因为输出可能会不符合要求
print(response)

{'title': '盗梦空间', 'year': 2010}
